# 07 — Train regression models

**Prepared, not run here:** reproducible Spark ML experiment for one-hour-ahead per-zone forecasting. Tune on chronological validation; no test data in fitting. The current January-only experiment is exploratory.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

## Study design

Target: `pickup_count` for a zone and hour, predicting after previous-hour actual counts are available. Historical features must be computed from pre-target observations. January 8–20 training, 21–25 validation, 26–31 test. Never use test outcomes for hyperparameter choice. The test window was previously inspected in the learning project; report it as non-pristine.

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

feat=spark.table("ml_hourly_demand_features")
needed=["lag_1","lag_2","lag_24","lag_168","rolling_mean_3h","rolling_mean_24h"]
model_data=feat.dropna(subset=needed)
train=model_data.filter(F.col("pickup_hour")<"2025-01-21")
valid=model_data.filter((F.col("pickup_hour")>="2025-01-21")&(F.col("pickup_hour")<"2025-01-26"))
test=model_data.filter(F.col("pickup_hour")>="2025-01-26")
print("Train/validation/test:",train.count(),valid.count(),test.count())
assert (train.count(),valid.count(),test.count())==(80808,31080,37296)

## Feature encoding

Treat zone and cyclical calendar positions as categories for linear modelling. Tree models use the same encoded feature vector for an apples-to-apples initial comparison. All encoders are fit using **training rows only**, inside the Pipeline.

In [ ]:
for col in ("PULocationID","hour_of_day","day_of_week"):
    train=train.withColumn(col+"_str",F.col(col).cast("string"))
    valid=valid.withColumn(col+"_str",F.col(col).cast("string"))
    test=test.withColumn(col+"_str",F.col(col).cast("string"))

categorical=["PULocationID_str","hour_of_day_str","day_of_week_str"]
indexers=[StringIndexer(inputCol=c,outputCol=c+"_index",handleInvalid="keep") for c in categorical]
encoder=OneHotEncoder(inputCols=[c+"_index" for c in categorical],outputCols=[c+"_ohe" for c in categorical],handleInvalid="keep")
numerical=["day_of_month","is_weekend","lag_1","lag_2","lag_24","lag_168","rolling_mean_3h","rolling_mean_24h"]
assembler=VectorAssembler(inputCols=[c+"_ohe" for c in categorical]+numerical,outputCol="features",handleInvalid="error")
preprocess=Pipeline(stages=indexers+[encoder,assembler]).fit(train)
train_vec=preprocess.transform(train).select("PULocationID","pickup_hour",F.col("pickup_count").cast("double").alias("label"),"features","lag_168")
valid_vec=preprocess.transform(valid).select("PULocationID","pickup_hour",F.col("pickup_count").cast("double").alias("label"),"features","lag_168")
test_vec=preprocess.transform(test).select("PULocationID","pickup_hour",F.col("pickup_count").cast("double").alias("label"),"features","lag_168")

## Fit candidate regressors and select on validation MAE

Metrics clip negative predictions to zero because taxi counts cannot be negative. Integer rounding is optional for deployment; for evaluation, we retain continuous nonnegative predictions.

In [ ]:
from pyspark.sql import functions as F

def predict_nonnegative(estimator, train_data, validation_data):
    fitted=estimator.fit(train_data)
    predictions=fitted.transform(validation_data).withColumn("prediction",F.greatest(F.lit(0.0),F.col("prediction")))
    mae=predictions.select(F.avg(F.abs(F.col("label")-F.col("prediction"))).alias("mae")).first()[0]
    return fitted,float(mae)

candidates={
    "linear_regression":LinearRegression(featuresCol="features",labelCol="label",regParam=0.1,elasticNetParam=0.0,maxIter=80),
    "random_forest":RandomForestRegressor(featuresCol="features",labelCol="label",numTrees=40,maxDepth=8,seed=42),
    "gbt":GBTRegressor(featuresCol="features",labelCol="label",maxIter=40,maxDepth=5,stepSize=0.1,seed=42),
}
models={}
validation_mae={}
for name,estimator in candidates.items():
    model,mae=predict_nonnegative(estimator,train_vec,valid_vec)
    models[name]=model
    validation_mae[name]=mae
    print(f"{name}: validation MAE={mae:.4f}")

baseline_mae=valid_vec.select(F.avg(F.abs(F.col("label")-F.col("lag_168")))).first()[0]
print(f"Previous-week baseline validation MAE: {baseline_mae:.4f}")
best_name=min(validation_mae,key=validation_mae.get)
print("Best ML candidate (VALIDATION ONLY):",best_name,"MAE",validation_mae[best_name])

## Evaluation by zone category on validation

Categories must be assigned using training-period data, not January-wide statistics.

In [ ]:
zone_groups=(train.groupBy("PULocationID").agg(F.avg("pickup_count").alias("train_mean"))
 .withColumn("demand_category",F.when(F.col("train_mean")<1,"Very Low").when(F.col("train_mean")<10,"Low").when(F.col("train_mean")<50,"Medium").otherwise("High")))
chosen=models[best_name].transform(valid_vec).withColumn("prediction",F.greatest(F.lit(0.0),F.col("prediction")))
comparison=(chosen.join(zone_groups.select("PULocationID","demand_category"),"PULocationID")
 .groupBy("demand_category").agg(F.count("*").alias("observations"),
 F.round(F.avg(F.abs(F.col("label")-F.col("prediction"))),4).alias("model_MAE"),
 F.round(F.avg(F.abs(F.col("label")-F.col("lag_168"))),4).alias("previous_week_MAE")))
comparison.show(truncate=False)

## Optional: test reporting (not an untouched holdout)

**Only** execute this cell after fixing the model design and documenting validation findings. More authoritative final results should use a fresh future month. No accuracy numbers are invented in this repository.

In [ ]:
# Because the original project has already inspected Jan 26–31 results,
# this is a retrospective test summary, NOT a pristine held-out evaluation.
selected_predictions=(models[best_name].transform(test_vec)
 .withColumn("prediction",F.greatest(F.lit(0.0),F.col("prediction"))))
selected_predictions.select(
 F.round(F.avg(F.abs(F.col("label")-F.col("prediction"))),4).alias("ML_test_MAE"),
 F.round(F.avg(F.abs(F.col("label")-F.col("lag_168"))),4).alias("weekly_baseline_test_MAE")
).show()